# Notebook 14 — Residual Scaling Law

This notebook continues the `prime-numbers-lab` sequence.

**Question.** Do normalized prime-gap distribution residuals decay with scale according to a measurable scaling law?

We test the empirical model

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha},
\]

where

\[
\Delta(z,x) = f_{\mathrm{emp}}(z,x) - e^{-z},
\qquad
z = \frac{\mathrm{gap}}{\log x}.
\]

The notebook keeps the same template pattern used in the earlier notebooks:

- reproducible setup
- generated figures saved into `../figures`
- tabular outputs saved into `../data`
- interpretation text saved into `../interpretations`
- compact summary metrics

In [ ]:

# ============================================================
# Notebook 14 — Residual Scaling Law
# ============================================================

import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RNG_SEED = 9423
rng = np.random.default_rng(RNG_SEED)

NOTEBOOK_ID = "14"
NOTEBOOK_SLUG = "residual_scaling_law"

# Repo-relative paths when run from notebooks/
FIG_DIR = Path("../figures")
DATA_DIR = Path("../data")
INTERP_DIR = Path("../interpretations")

for d in [FIG_DIR, DATA_DIR, INTERP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def savefig(name, dpi=160):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"saved: {path}")
    return path

def save_csv(df, name):
    path = DATA_DIR / f"{NOTEBOOK_ID}_{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path}")
    return path

def display_df(df, n=10):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string(index=False))

## 1. Prime generation and gap normalization

We generate primes up to a finite scale and compute consecutive prime gaps.

For each prime \(p_n\), with gap

\[
g_n = p_{n+1} - p_n,
\]

we normalize by the local logarithmic scale:

\[
z_n = \frac{g_n}{\log p_n}.
\]

If the standard exponential-spacing heuristic becomes more accurate at large scale, then \(z_n\) should move closer to an \(\mathrm{Exp}(1)\) distribution.

In [ ]:

def sieve_primes(n: int) -> np.ndarray:
    # Return all primes <= n using a standard boolean sieve.
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    max_i = int(n**0.5)
    for i in range(2, max_i + 1):
        if sieve[i]:
            sieve[i*i:n+1:i] = False
    return np.nonzero(sieve)[0].astype(np.int64)

# Larger than earlier notebooks but still Colab-friendly.
N_MAX = 2_000_000

primes = sieve_primes(N_MAX)
p_left = primes[:-1].astype(float)
gaps = np.diff(primes).astype(float)
logp = np.log(p_left)
z = gaps / logp

prime_gap_df = pd.DataFrame({
    "p": p_left.astype(np.int64),
    "next_p": primes[1:].astype(np.int64),
    "gap": gaps.astype(np.int64),
    "log_p": logp,
    "z": z,
})

print(f"N_MAX = {N_MAX:,}")
print(f"prime count = {len(primes):,}")
print(f"gap count = {len(gaps):,}")
display_df(prime_gap_df)

## 2. Logarithmic scale windows

We split primes into logarithmic \(x\)-windows.  
Each window gets its own empirical normalized-gap distribution.

For each window, we compute:

\[
f_{\mathrm{emp}}(z,x)
\]

and compare it to

\[
f_{\mathrm{Exp}(1)}(z)=e^{-z}.
\]

In [ ]:

# Windowing choices. Keep enough windows for a visible scaling trend.
X_MIN = 3_000
X_MAX = float(p_left.max())
N_WINDOWS = 14

window_edges = np.geomspace(X_MIN, X_MAX, N_WINDOWS + 1)

# PDF grid for normalized gaps.
Z_MIN, Z_MAX = 0.0, 6.0
N_BINS = 80
z_edges = np.linspace(Z_MIN, Z_MAX, N_BINS + 1)
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
dz = z_edges[1] - z_edges[0]

def exp_pdf(z_values):
    return np.exp(-z_values)

exp_vals = exp_pdf(z_centers)

window_rows = []
pdf_rows = []

for i in range(N_WINDOWS):
    lo, hi = window_edges[i], window_edges[i + 1]
    mask = (p_left >= lo) & (p_left < hi)
    zw = z[mask]
    pw = p_left[mask]
    gw = gaps[mask]

    if len(zw) < 20:
        continue

    hist_density, _ = np.histogram(zw, bins=z_edges, density=True)
    delta = hist_density - exp_vals

    l1 = float(np.sum(np.abs(delta)) * dz)
    l2 = float(np.sqrt(np.sum(delta**2) * dz))
    pos_mass = float(np.sum(np.clip(delta, 0, None)) * dz)
    neg_mass = float(np.sum(np.clip(-delta, 0, None)) * dz)

    emp_prob, _ = np.histogram(zw, bins=z_edges, density=False)
    emp_prob = emp_prob.astype(float)
    emp_prob = emp_prob / emp_prob.sum()
    exp_prob = exp_vals * dz
    exp_prob = exp_prob / exp_prob.sum()

    m = 0.5 * (emp_prob + exp_prob)
    eps = 1e-15
    kl_emp = np.sum(emp_prob * np.log((emp_prob + eps) / (m + eps)))
    kl_exp = np.sum(exp_prob * np.log((exp_prob + eps) / (m + eps)))
    js = float(0.5 * (kl_emp + kl_exp))

    mean_z = float(np.mean(zw))
    std_z = float(np.std(zw))
    q50, q75, q90, q95, q99 = np.quantile(zw, [0.50, 0.75, 0.90, 0.95, 0.99])

    signed_mass = float(np.sum(delta) * dz)
    tail_mask = z_centers >= 3.0
    tail_bias = float(np.sum(delta[tail_mask]) * dz)

    midpoint = float(np.sqrt(lo * hi))

    window_rows.append({
        "window_index": i,
        "x_lo": lo,
        "x_hi": hi,
        "x_mid": midpoint,
        "n_gaps": int(len(zw)),
        "mean_gap": float(np.mean(gw)),
        "mean_logp": float(np.mean(np.log(pw))),
        "mean_z": mean_z,
        "std_z": std_z,
        "q50_z": float(q50),
        "q75_z": float(q75),
        "q90_z": float(q90),
        "q95_z": float(q95),
        "q99_z": float(q99),
        "l1_residual": l1,
        "l2_residual": l2,
        "js_divergence": js,
        "positive_residual_mass": pos_mass,
        "negative_residual_mass": neg_mass,
        "signed_residual_mass": signed_mass,
        "tail_bias_z_ge_3": tail_bias,
    })

    for zc, pdf, ev, de in zip(z_centers, hist_density, exp_vals, delta):
        pdf_rows.append({
            "window_index": i,
            "x_mid": midpoint,
            "z": float(zc),
            "empirical_pdf": float(pdf),
            "exp1_pdf": float(ev),
            "delta": float(de),
        })

metrics_df = pd.DataFrame(window_rows)
pdf_df = pd.DataFrame(pdf_rows)

save_csv(metrics_df, "residual_scaling_metrics")
save_csv(pdf_df, "residual_pdf_grid")

display_df(metrics_df, n=20)

## 3. Residual norms versus scale

We first plot residual norms directly against scale.

If convergence is present, we expect residual magnitudes to trend downward as \(x\) increases.

In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(metrics_df["x_mid"], metrics_df["l1_residual"], marker="o", label="L1 residual")
plt.plot(metrics_df["x_mid"], metrics_df["l2_residual"], marker="o", label="L2 residual")
plt.xscale("log")
plt.xlabel("window midpoint x")
plt.ylabel("residual norm")
plt.title("Residual norm vs scale")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("residual_norm_vs_scale")
plt.show()

## 4. Scaling-law fit

We fit

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}.
\]

Taking logarithms gives

\[
\log \|\Delta(z,x)\| = \log C - \alpha \log\log x.
\]

So the fitted line has slope \(-\alpha\).

In [ ]:

def fit_scaling_law(df, y_col):
    fit_df = df[["x_mid", y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    fit_df = fit_df[fit_df[y_col] > 0].copy()

    X = np.log(np.log(fit_df["x_mid"].values))
    Y = np.log(fit_df[y_col].values)

    coeff = np.polyfit(X, Y, 1)
    slope, intercept = coeff[0], coeff[1]
    yhat = slope * X + intercept

    ss_res = float(np.sum((Y - yhat) ** 2))
    ss_tot = float(np.sum((Y - np.mean(Y)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    alpha = float(-slope)
    C = float(np.exp(intercept))

    return {
        "metric": y_col,
        "alpha": alpha,
        "C": C,
        "slope": float(slope),
        "intercept": float(intercept),
        "r2": r2,
        "n_fit": int(len(fit_df)),
    }, fit_df, X, Y, yhat

fit_results = []
fit_cache = {}

for col in ["l1_residual", "l2_residual", "js_divergence"]:
    result, fit_df, X, Y, yhat = fit_scaling_law(metrics_df, col)
    fit_results.append(result)
    fit_cache[col] = (fit_df, X, Y, yhat, result)

fit_summary_df = pd.DataFrame(fit_results)
save_csv(fit_summary_df, "scaling_fit_summary")
display_df(fit_summary_df, n=10)

In [ ]:

def plot_fit(metric_col, label, filename):
    fit_df, X, Y, yhat, result = fit_cache[metric_col]
    x_grid = np.geomspace(fit_df["x_mid"].min(), fit_df["x_mid"].max(), 200)
    model = result["C"] * (np.log(x_grid) ** (-result["alpha"]))

    plt.figure(figsize=(10, 6))
    plt.plot(fit_df["x_mid"], fit_df[metric_col], marker="o", linestyle="", label=f"observed {label}")
    plt.plot(x_grid, model, label=f"fit: alpha={result['alpha']:.3f}, R²={result['r2']:.4f}")
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("window midpoint x")
    plt.ylabel(label)
    plt.title(f"{label} scaling fit")
    plt.grid(True, which="both", alpha=0.35)
    plt.legend()
    savefig(filename)
    plt.show()

plot_fit("l1_residual", "L1 residual", "l1_scaling_fit")
plot_fit("l2_residual", "L2 residual", "l2_scaling_fit")
plot_fit("js_divergence", "JS divergence", "js_scaling_fit")

## 5. Alpha comparison

The fitted exponents summarize how quickly each finite-scale residual metric decays.

Larger \(\alpha\) means faster decay in the model

\[
\|\Delta\| \sim C(\log x)^{-\alpha}.
\]

In [ ]:

plt.figure(figsize=(9, 6))
plt.bar(fit_summary_df["metric"], fit_summary_df["alpha"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("fitted alpha")
plt.xlabel("residual metric")
plt.title("Scaling exponent alpha comparison")
plt.grid(True, axis="y", alpha=0.35)
savefig("alpha_comparison")
plt.show()

## 6. Residuals of fitted scaling law

A scaling law is useful only if its fit residuals are not dominated by obvious unresolved structure.

We therefore plot

\[
\log(\|\Delta\|) - \log(\widehat{\|\Delta\|})
\]

against scale.

In [ ]:

plt.figure(figsize=(10, 6))

for metric_col in ["l1_residual", "l2_residual", "js_divergence"]:
    fit_df, X, Y, yhat, result = fit_cache[metric_col]
    resid = Y - yhat
    plt.plot(fit_df["x_mid"], resid, marker="o", label=metric_col)

plt.axhline(0, linestyle="--")
plt.xscale("log")
plt.xlabel("window midpoint x")
plt.ylabel("log-fit residual")
plt.title("Scaling-law fit residuals")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("fit_residuals")
plt.show()

## 7. Residual heatmap across scale

Notebook 13 showed residual structure across normalized gap values.  
Here we keep that view and add scale-law context.

The residual is

\[
\Delta(z,x)=f_{\mathrm{emp}}(z,x)-e^{-z}.
\]

In [ ]:

pivot = pdf_df.pivot_table(index="window_index", columns="z", values="delta", aggfunc="mean")
ordered_windows = metrics_df.sort_values("x_mid")["window_index"].tolist()
pivot = pivot.loc[ordered_windows]

plt.figure(figsize=(10, 7))
extent = [z_centers.min(), z_centers.max(), 0, len(pivot)]
plt.imshow(
    pivot.values,
    aspect="auto",
    origin="lower",
    extent=extent,
    interpolation="nearest"
)
plt.colorbar(label="Delta(z, x)")
plt.axvline(1.0, linestyle="--", alpha=0.7)
plt.axvline(3.0, linestyle="--", alpha=0.7)
plt.xlabel("normalized gap z")
plt.ylabel("window index")
plt.title("Residual heatmap across scale")
savefig("residual_heatmap_across_scale")
plt.show()

## 8. Positive and negative residual mass

The residual has sign. We compare positive mass against negative mass magnitude:

\[
\int \max(\Delta,0)\,dz
\]

and

\[
\int \max(-\Delta,0)\,dz.
\]

In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(metrics_df["x_mid"], metrics_df["positive_residual_mass"], marker="o", label="positive residual mass")
plt.plot(metrics_df["x_mid"], metrics_df["negative_residual_mass"], marker="o", label="negative residual mass magnitude")
plt.xscale("log")
plt.xlabel("window midpoint x")
plt.ylabel("mass")
plt.title("Positive / negative residual mass")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("positive_negative_residual_mass")
plt.show()

## 9. Distributional convergence score

A compact score is useful for the repo overview.

We define:

\[
S(x)=\frac{1}{1+\mathrm{L1}(x)+\mathrm{L2}(x)+\mathrm{JS}(x)}.
\]

This is not a theorem. It is an empirical diagnostic: higher means closer to the exponential baseline under the chosen finite-window measurement.

In [ ]:

metrics_df["distributional_convergence_score"] = 1.0 / (
    1.0
    + metrics_df["l1_residual"]
    + metrics_df["l2_residual"]
    + metrics_df["js_divergence"]
)

plt.figure(figsize=(10, 6))
plt.plot(metrics_df["x_mid"], metrics_df["distributional_convergence_score"], marker="o", label="convergence score")
plt.xscale("log")
plt.xlabel("window midpoint x")
plt.ylabel("score")
plt.title("Distributional convergence score by scale")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("distributional_convergence_score")
plt.show()

save_csv(metrics_df, "residual_scaling_metrics_with_score")
display_df(metrics_df[[
    "window_index", "x_mid", "n_gaps",
    "l1_residual", "l2_residual", "js_divergence",
    "distributional_convergence_score"
]], n=20)

## 10. Tail residual bias

We also track residual bias in the large-gap tail:

\[
\int_{z\geq 3}\Delta(z,x)\,dz.
\]

This helps identify whether finite-scale deviations come mainly from tail excess, tail deficit, or small-gap structure.

In [ ]:

plt.figure(figsize=(10, 6))
plt.plot(metrics_df["x_mid"], metrics_df["tail_bias_z_ge_3"], marker="o", label="tail residual bias z >= 3")
plt.axhline(0, linestyle="--")
plt.xscale("log")
plt.xlabel("window midpoint x")
plt.ylabel("tail bias")
plt.title("Tail residual bias for z >= 3")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("tail_residual_bias_z_ge_3")
plt.show()

## 11. Final window check

The largest window gives the most asymptotic-looking finite sample in this notebook.  
We compare its empirical PDF against \(\mathrm{Exp}(1)\).

In [ ]:

last_idx = int(metrics_df.sort_values("x_mid").iloc[-1]["window_index"])
last_pdf = pdf_df[pdf_df["window_index"] == last_idx].copy()

plt.figure(figsize=(10, 6))
plt.plot(last_pdf["z"], last_pdf["empirical_pdf"], marker="o", label="empirical PDF")
plt.plot(last_pdf["z"], last_pdf["exp1_pdf"], linewidth=2, label="Exp(1)")
plt.xlabel("normalized gap z")
plt.ylabel("density")
plt.title("Final window PDF vs Exp(1)")
plt.grid(True, alpha=0.35)
plt.legend()
savefig("final_window_pdf_vs_exp1")
plt.show()

## 12. Interpretation

Notebook 14 makes Notebook 13's residual observation measurable.

Core empirical result:

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}.
\]

Interpretation:

- normalized prime-gap residuals decay with scale
- L1, L2, and JS residuals provide independent diagnostics
- fitted \(\alpha>0\) supports finite-scale convergence toward the exponential spacing baseline
- remaining fit residuals indicate finite-window structure still exists
- Notebook 15 should decompose residuals by residue classes or local arithmetic structure

Conservative claim:

> Finite-scale normalized prime-gap distributions move toward the exponential baseline, and their residual size can be summarized by a log-scale decay law.

Constraint → signal > noise.

In [ ]:

interpretation_md = f"""# Notebook 14 — Residual Scaling Law

## Core question

Notebook 13 showed that normalized prime-gap residuals shrink with scale.

Notebook 14 asks whether that shrinkage follows a measurable scaling law.

We model:

```tex
||Δ(z,x)|| ~ C (log x)^(-α)
```

where:

```tex
Δ(z,x) = f_emp(z,x) - exp(-z)
z = gap / log(x)
```

---

## Main diagnostics

The notebook measures residual size by:

- L1 residual norm
- L2 residual norm
- Jensen-Shannon divergence
- positive residual mass
- negative residual mass
- tail residual bias for z >= 3

---

## Fitted scaling law

Log-linear fit:

```tex
log ||Δ|| = log C - α log log x
```

Fit summary:

{fit_summary_df.to_markdown(index=False)}

---

## Main figures

![Residual norm vs scale](../figures/14_residual_norm_vs_scale.png)

![L1 scaling fit](../figures/14_l1_scaling_fit.png)

![L2 scaling fit](../figures/14_l2_scaling_fit.png)

![JS scaling fit](../figures/14_js_scaling_fit.png)

![Alpha comparison](../figures/14_alpha_comparison.png)

![Scaling-law fit residuals](../figures/14_fit_residuals.png)

![Residual heatmap across scale](../figures/14_residual_heatmap_across_scale.png)

![Positive and negative residual mass](../figures/14_positive_negative_residual_mass.png)

![Distributional convergence score](../figures/14_distributional_convergence_score.png)

![Tail residual bias](../figures/14_tail_residual_bias_z_ge_3.png)

![Final window PDF vs Exp(1)](../figures/14_final_window_pdf_vs_exp1.png)

---

## Interpretation

Normalized prime-gap residuals decrease with scale.

The fitted exponent α gives a compact empirical measurement of convergence speed under this finite-window diagnostic.

This does not prove an asymptotic theorem. It gives a reproducible finite-scale measurement pipeline:

```tex
finite gaps -> normalized gaps -> empirical PDF -> residual Δ(z,x) -> scaling fit
```

---

## Conservative result

Finite-scale normalized prime-gap distributions move toward the exponential baseline, and their residual size can be summarized by a log-scale decay law.

---

Constraint → signal > noise
"""

interp_path = INTERP_DIR / "14_residual_scaling_law.md"
interp_path.write_text(interpretation_md, encoding="utf-8")
print(f"saved: {interp_path}")
print(interpretation_md[:1200])

## 13. Repo commit checklist

Suggested files to commit after running this notebook:

```bash
git add notebooks/14_residual_scaling_law.ipynb \
        figures/14_*.png \
        data/14_*.csv \
        interpretations/14_residual_scaling_law.md

git commit -m "Add Notebook 14 residual scaling law"
git push
```